<a href="https://colab.research.google.com/github/nilsugungor/potsdam-hackathon/blob/main/2022_model_gpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "gpt2"

print("Model yükleniyor (Hata riskine karşı CPU/GPU otomatik seçiliyor)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)

print("Success!.")

prompt = """The year is 2040. The world has transformed into a landscape of
shimmering heat and altered geography. New continents have emerged where
the old ones sank, and humanity has adapted in strange, beautiful ways.
The story begins in the city of Neo-Pangea, where the air tastes like
salt and electricity.

Chapter 1: The Glass Horizon

Elias stood on the edge of the terrace, looking out at the submerged
remains of the old world."""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=700, do_sample=True)
text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Output ---\n", text)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

full_text = """ Instruction: Write a long essay about the year 2040.
Output: The year 2040 will be a turning point for humanity. First,

Instruction:  Write the first chapter of a science fiction novel. The story is set on a future Earth with changed climate and geography. Begin with a description of the planet and its history. Then describe the world in detail: the land, the cities, the people, and how they live. Next, show a public scene where a character observes something that makes them question what they have been told. The character thinks carefully about what they see.

Output: In the coming decades, the Earth will face unprecedented challenges...
"""
target_word_count = 5000
current_word_count = len(full_text.split())

print(f"Starting generation... Current count: {current_word_count} words.")

while current_word_count < target_word_count:
    context = " ".join(full_text.split()[-300:])
    inputs = tokenizer(context, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    new_chunk = tokenizer.decode(outputs[0], skip_special_tokens=True)

    generated_only = new_chunk[len(context):].strip()

    full_text += " " + generated_only
    current_word_count = len(full_text.split())

    print(f"Progress: {current_word_count} / {target_word_count} words...")

with open("novel_chapter_1.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print("✅ Novel chapter saved to novel_chapter_1.txt")

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files

model_id = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

full_text = """The year 2040 arrived with a shimmering, heat-distorted silence.
The old maps were useless now; the oceans had reclaimed the coastlines,
carving new shapes into the continents. In the heart of the new world,
cities of recycled glass rose from the tides. People lived in
vertical forests, shielding themselves from the relentless sun.

Elias stood on the high balcony of Neo-Pangea, looking down at the
emerald waters that now flowed through what used to be Central Park.
Everything the Council told them about the Great Stabilization felt
like a dream, or perhaps, a well-crafted lie. He noticed a
flicker of light beneath the waves that shouldn't be there."""

target_word_count = 5000
current_word_count = len(full_text.split())

print(f"Başlıyoruz. Şu anki kelime sayısı: {current_word_count}")

while current_word_count < target_word_count:
    context_words = full_text.split()[-300:]
    context = " ".join(context_words)

    inputs = tokenizer(context, return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[1]

    if input_len > 850:
        inputs.input_ids = inputs.input_ids[:, -850:]
        input_len = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.85,
            top_p=0.92,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_len:]
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    full_text += " " + generated_text
    current_word_count = len(full_text.split())

    print(f"Progress: {current_word_count} / {target_word_count} words...")
    if current_word_count % 500 <= 150 and current_word_count > 500:
        with open("novel_backup.txt", "w", encoding="utf-8") as f:
            f.write(full_text)

filename = "novel_chapter_2040_final.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write(full_text)

print(f"Success! Toplam {current_word_count} kelime.")
files.download(filename)